# Figure 6 — Amphibole compositions and aluminum site occupancy


## 1. Imports and shared 00_Setup

This notebook uses Arial, 8-point ordinary text, 9-point bold uppercase panel labels, full axes boxes, light gridlines, 300 dpi PNG export, and editable PDF/SVG text. The code is linear and contains no custom functions.


In [1]:
from pathlib import Path
import hashlib
import inspect
import re

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from packaging.version import Version
from importlib.metadata import PackageNotFoundError, version as package_version
import numpy as np
import pandas as pd
from PIL import Image
import Thermobar
import Thermobar.core as tb_core
import Thermobar.amphibole as tb_amphibole

SINGLE_COLUMN_WIDTH_IN = 3.55
DOUBLE_COLUMN_WIDTH_IN = 7.25
MAXIMUM_FIGURE_HEIGHT_IN = 7.8
FIGURE_WIDTH_IN = DOUBLE_COLUMN_WIDTH_IN
FIGURE_HEIGHT_IN = 5.15
BASE_TEXT_SIZE_PT = 8.0
AXIS_LABEL_SIZE_PT = 8.0
PANEL_LABEL_SIZE_PT = 9.0
LEGEND_SIZE_PT = 6.0
MINIMUM_LINE_WIDTH_PT = 0.28
EXPORT_DPI = 300
PUBLICATION_FONT = "Arial"

installed_font_names = {entry.name for entry in font_manager.fontManager.ttflist}
if PUBLICATION_FONT not in installed_font_names:
    raise RuntimeError("Arial is required by 00_Setup.")
mpl.rcParams.update({
    "font.family":"sans-serif", "font.sans-serif":[PUBLICATION_FONT],
    "font.size":BASE_TEXT_SIZE_PT, "axes.labelsize":AXIS_LABEL_SIZE_PT,
    "axes.titlesize":BASE_TEXT_SIZE_PT, "xtick.labelsize":BASE_TEXT_SIZE_PT,
    "ytick.labelsize":BASE_TEXT_SIZE_PT, "legend.fontsize":LEGEND_SIZE_PT,
    "figure.facecolor":"white", "axes.facecolor":"white",
    "axes.spines.top":True, "axes.spines.right":True, "axes.linewidth":0.5,
    "lines.linewidth":0.75, "xtick.direction":"in", "ytick.direction":"in",
    "xtick.major.width":0.5, "ytick.major.width":0.5,
    "savefig.dpi":EXPORT_DPI, "savefig.facecolor":"white",
    "savefig.transparent":False, "pdf.fonttype":42, "ps.fonttype":42,
    "svg.fonttype":"none",
})
THERMOBAR_VERSION = package_version("Thermobar") if package_version("Thermobar") else getattr(Thermobar, "__version__", "unknown")


## 2. Paths and public S1–S3 workbooks

Figure 6 uses the public S1 Transects and Mingled_Minerals sheets. S2 and S3 are checked as part of the shared package but are not used for the plotted populations.


In [2]:
PROJECT_ROOT_OVERRIDE = None
PROJECT_ROOT = PROJECT_ROOT_OVERRIDE
if PROJECT_ROOT is None:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "SA_MI_Draft_FinalFigures_Work" / "DataSheets").is_dir():
            PROJECT_ROOT = candidate
            break
if PROJECT_ROOT is None:
    raise FileNotFoundError("Run this notebook from the project or set PROJECT_ROOT_OVERRIDE.")
PROJECT_ROOT = Path(PROJECT_ROOT).expanduser().resolve()
WORK_ROOT = PROJECT_ROOT / "SA_MI_Draft_FinalFigures_Work"
DATA_DIRECTORY = WORK_ROOT / "DataSheets" / "Updated_Oct20"
FIGURE_DIRECTORY = WORK_ROOT / "Nov6_AllFigures" / "Figure_06_Amphiboles"
OUTPUT_DIRECTORY = FIGURE_DIRECTORY / "Figure_06_Amphiboles_Final"
OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)

S1_PATH = DATA_DIRECTORY / "Dechert_S1_AllData.xlsx"
S2_PATH = DATA_DIRECTORY / "Dechert_S2_Standards.xlsx"
S3_PATH = DATA_DIRECTORY / "Dechert_S3_Compilations.xlsx"
for workbook_path in [S1_PATH, S2_PATH, S3_PATH]:
    if not workbook_path.is_file():
        raise FileNotFoundError(f"Missing supplemental workbook: {workbook_path}")
SOURCE_HASHES_BEFORE = {
    "S1":hashlib.sha256(S1_PATH.read_bytes()).hexdigest(),
    "S2":hashlib.sha256(S2_PATH.read_bytes()).hexdigest(),
    "S3":hashlib.sha256(S3_PATH.read_bytes()).hexdigest(),
}


## 3. Read S1 and configure the main-pumice transect and mingled-pumice populations


In [3]:
transects_source = pd.read_excel(S1_PATH, sheet_name="Transects")
hosts_source = pd.read_excel(S1_PATH, sheet_name="MI_Hosts_Plag_Pxy")
mingled_source = pd.read_excel(S1_PATH, sheet_name="Mingled_Minerals")

all_linescans = transects_source.rename(columns={"Sample":"Sample_Group"}).copy()
scan_counts = all_linescans.groupby("Linescan_ID")["Predict_Mineral"].value_counts(dropna=False)
scan_total = all_linescans.groupby("Linescan_ID").size()
dominant_table = (
    scan_counts.rename("n").reset_index()
    .sort_values(["Linescan_ID","n"], ascending=[True,False], kind="stable")
    .drop_duplicates("Linescan_ID")
)
dominant = dominant_table.set_index("Linescan_ID")["Predict_Mineral"]
fraction = scan_counts.groupby(level=0).max().div(scan_total)
all_linescans["Dominant_Mineral"] = all_linescans["Linescan_ID"].map(dominant)
all_linescans["Dominant_Mineral_Fraction"] = all_linescans["Linescan_ID"].map(fraction)
assert all_linescans["Sample_Group"].isin(["74A","74D","75","76A"]).all()
assert fraction.ge(0.95 - 1e-12).all()
assert not all_linescans["Source_CSV"].astype("string").str.contains("75_1_C32|74A2_amph9|^76B", case=False, regex=True, na=False).any()
mingled_minerals = mingled_source.rename(columns={"Type of Pumice":"Mafic/Felsic"}).copy()
dechert_combined_data = hosts_source.copy()
measured_legacy = {"SiO2":"Si Ox%", "TiO2":"Ti Ox%", "Al2O3":"Al Ox%", "FeOt":"Fe Ox%", "MnO":"Mn Ox%", "MgO":"Mg Ox%", "CaO":"Ca Ox%", "Na2O":"Na Ox%", "K2O":"K Ox%", "Cl":"Cl Ox%"}
for oxide, legacy_column in measured_legacy.items():
    all_linescans[legacy_column] = all_linescans[f"{oxide}_SEMMineral_Raw"]

AMPHIBOLE_POOL_MEMBERS = {"B":["74D","74A"], "G":["75","76A"]}
AMPHIBOLE_POOL_ORDER = ["B","G"]
AMPHIBOLE_POOL_LINE_CMAPS = {"B":"Oranges", "G":"Blues"}
AMPHIBOLE_POOL_HIST_COLORS = {"B":"#F4A261", "G":"#0072B2"}
AMPHIBOLE_MINGLED_DATASET = {"B":"74Ming", "G":"76Ming"}
MINGLED_AMPHIBOLE_COLORS = {
    "B":{"Light":"#EE3377","Dark":"#808080"},
    "G":{"Light":"#66C2C2","Dark":"#000000"},
}
MINGLED_DISPLAY_MAP = {"felsic":"Light","mafic":"Dark","light":"Light","dark":"Dark"}
HIGH_AL2O3_LINE_IDS = {
    "74A2_amph12_linescan_lr":"74A2_amph 12",
    "74A2_amph7_linescan_rl":"74A2_amph 7",
    "74A2_px14_linescan_lr":"74A2 px14",
}
HIGH_AL2O3_AMPHIBOLE_COLOR = "#39FF14"
MML_PREDICTION_SCORE_MIN = 0.85
MINERAL_DOMINANCE_MIN = 0.95
HIST_OVERSAMPLE_MULTIPLIER = 10
N_HISTOGRAM_EDGES = 20
TRANSECT_SITE_MARKER = "^"
MINGLED_SITE_MARKER = "^"
TRANSECT_SITE_POINT_SIZE = 22.0
MINGLED_SITE_POINT_SIZE = 34.0
AMPHIBOLE_POINT_EDGE_WIDTH = 0.28
RAW_OXIDE_MAP = {
    "SiO2":"Si Ox%", "TiO2":"Ti Ox%", "Al2O3":"Al Ox%",
    "FeOt":"Fe Ox%", "MnO":"Mn Ox%", "MgO":"Mg Ox%",
    "CaO":"Ca Ox%", "Na2O":"Na Ox%", "K2O":"K Ox%",
}
MINGLED_RAW_OXIDE_MAP = {
    "SiO2":"SiO2_SEMMineral_Raw", "TiO2":"TiO2_SEMMineral_Raw",
    "Al2O3":"Al2O3_SEMMineral_Raw", "FeOt":"FeOt_SEMMineral_Raw",
    "MnO":"MnO_SEMMineral_Raw", "MgO":"MgO_SEMMineral_Raw",
    "CaO":"CaO_SEMMineral_Raw", "Na2O":"Na2O_SEMMineral_Raw",
    "K2O":"K2O_SEMMineral_Raw", "Cr2O3":"Cr2O3_SEMMineral_Raw",
}


## 4. Filter main-pumice transects and mingled-pumice amphibole point analyses


In [4]:
linescan_work = all_linescans.copy()
linescan_work["_distance_um"] = pd.to_numeric(linescan_work["Distance (µm)"], errors="coerce")
linescan_work["_Al2O3_norm"] = pd.to_numeric(linescan_work["Al2O3_SEMMineral_Norm"], errors="coerce")
linescan_work["_prediction_score"] = pd.to_numeric(linescan_work["Prediction_Score"], errors="coerce")
linescan_work["_dominant_fraction"] = pd.to_numeric(linescan_work["Dominant_Mineral_Fraction"], errors="coerce")
standard_pattern = r"(?:albite|kl_?2g?|macfo|anorthite|labradorite|lab\d)"
linescan_work["_is_standard"] = linescan_work["Source_CSV"].astype("string").str.contains(standard_pattern, case=False, regex=True, na=False)
linescan_work["_is_discarded"] = False
for discard_column in ["discard","Discard"]:
    if discard_column in linescan_work.columns:
        linescan_work["_is_discarded"] = linescan_work["_is_discarded"] | linescan_work[discard_column].astype("string").str.strip().str.casefold().isin(["1","true","yes","y","discard","discarded"]).fillna(False)
nonstandard_nondiscarded = ~linescan_work["_is_standard"] & ~linescan_work["_is_discarded"]
all_pool_members = {"74A","74D","75","76A"}
amphibole_mask = (
    nonstandard_nondiscarded
    & linescan_work["Sample_Group"].isin(all_pool_members)
    & linescan_work["Dominant_Mineral"].astype("string").str.strip().str.casefold().eq("amphibole")
    & linescan_work["_dominant_fraction"].ge(MINERAL_DOMINANCE_MIN)
    & linescan_work["Predict_Mineral"].astype("string").str.strip().str.casefold().eq("amphibole")
    & linescan_work["_prediction_score"].gt(MML_PREDICTION_SCORE_MIN)
    & linescan_work["_distance_um"].notna()
    & linescan_work["_Al2O3_norm"].notna()
)
amphibole_linescan_points = linescan_work.loc[amphibole_mask].copy()

AMPHIBOLE_TRANSECTS = {}
TRANSECT_COLORS = {}
for pool in AMPHIBOLE_POOL_ORDER:
    pool_transects = []
    samples = AMPHIBOLE_POOL_MEMBERS[pool]
    for sample in samples:
        sample_frame = amphibole_linescan_points.loc[amphibole_linescan_points["Sample_Group"].eq(sample)]
        for linescan_id, transect in sample_frame.groupby("Linescan_ID", sort=True):
            plotted = transect.sort_values(["_distance_um","Point_Order"], kind="stable").copy()
            if len(plotted) > 0:
                plotted.attrs["linescan_id"] = linescan_id
                plotted.attrs["sample_group"] = sample
                pool_transects.append(plotted)
    AMPHIBOLE_TRANSECTS[pool] = pool_transects
    color_positions = np.linspace(0.48,0.92,max(len(pool_transects),2))
    cmap = plt.colormaps[AMPHIBOLE_POOL_LINE_CMAPS[pool]]
    for index, transect in enumerate(pool_transects):
        linescan_id = transect.attrs["linescan_id"]
        TRANSECT_COLORS[linescan_id] = mpl.colors.to_hex(cmap(color_positions[index]), keep_alpha=False)
for linescan_id in HIGH_AL2O3_LINE_IDS:
    TRANSECT_COLORS[linescan_id] = HIGH_AL2O3_AMPHIBOLE_COLOR

mingled_prediction_score = pd.to_numeric(mingled_minerals["Prediction_Score"], errors="coerce")
mingled_amphibole_mask = (
    mingled_minerals["Predict_Mineral"].astype("string").str.strip().str.casefold().eq("amphibole")
    & mingled_prediction_score.gt(MML_PREDICTION_SCORE_MIN)
    & mingled_minerals["Dataset"].isin(AMPHIBOLE_MINGLED_DATASET.values())
)
mingled_amphibole_plot_data = mingled_minerals.loc[mingled_amphibole_mask].copy()
mingled_amphibole_plot_data["_Al2O3_norm"] = pd.to_numeric(mingled_amphibole_plot_data["Al2O3_SEMMineral_Norm"], errors="coerce")
mingled_amphibole_plot_data["_display_texture"] = mingled_amphibole_plot_data["Mafic/Felsic"].astype("string").str.strip().str.casefold().map(MINGLED_DISPLAY_MAP)
mingled_amphibole_plot_data = mingled_amphibole_plot_data.dropna(subset=["Dataset","_display_texture","_Al2O3_norm"]).reset_index(drop=True)
assert set(HIGH_AL2O3_LINE_IDS).issubset(TRANSECT_COLORS)
assert set(mingled_amphibole_plot_data["_display_texture"].unique()) <= {"Light","Dark"}


## 5. Equal-weight transect histograms and aluminum site allocation


In [5]:
HIST_MAX_ORIGINAL_POINTS = int(linescan_work.loc[nonstandard_nondiscarded].groupby("Linescan_ID")["_distance_um"].count().max())
HIST_N_RESAMPLE = HIST_MAX_ORIGINAL_POINTS * HIST_OVERSAMPLE_MULTIPLIER
AMPHIBOLE_EQUAL_WEIGHT_VALUES = {}
for pool in AMPHIBOLE_POOL_ORDER:
    interpolated = []
    for transect in AMPHIBOLE_TRANSECTS[pool]:
        xy = transect[["_distance_um","_Al2O3_norm"]].dropna().groupby("_distance_um", as_index=False)["_Al2O3_norm"].mean().sort_values("_distance_um")
        if len(xy) == 1:
            interpolated.append(np.repeat(float(xy["_Al2O3_norm"].iloc[0]), HIST_N_RESAMPLE))
        elif len(xy) > 1:
            interpolated.append(np.interp(
                np.linspace(0.0,1.0,HIST_N_RESAMPLE),
                np.linspace(0.0,1.0,len(xy)),
                xy["_Al2O3_norm"].to_numpy(float),
            ))
    AMPHIBOLE_EQUAL_WEIGHT_VALUES[pool] = np.concatenate(interpolated) if interpolated else np.array([], dtype=float)
AMPHIBOLE_MINGLED_VALUES = {
    pool:{
        texture:mingled_amphibole_plot_data.loc[
            mingled_amphibole_plot_data["Dataset"].eq(AMPHIBOLE_MINGLED_DATASET[pool])
            & mingled_amphibole_plot_data["_display_texture"].eq(texture),
            "_Al2O3_norm",
        ].to_numpy(float)
        for texture in ["Light","Dark"]
    }
    for pool in AMPHIBOLE_POOL_ORDER
}
all_al2o3_values = [
    *(AMPHIBOLE_EQUAL_WEIGHT_VALUES[pool] for pool in AMPHIBOLE_POOL_ORDER),
    *(AMPHIBOLE_MINGLED_VALUES[pool][texture] for pool in AMPHIBOLE_POOL_ORDER for texture in ["Light","Dark"]),
]
all_al2o3_values = [values for values in all_al2o3_values if len(values)]
AL2O3_MIN = min(float(np.nanmin(values)) for values in all_al2o3_values)
AL2O3_MAX = max(float(np.nanmax(values)) for values in all_al2o3_values)
AL2O3_BINS = np.linspace(AL2O3_MIN, AL2O3_MAX, N_HISTOGRAM_EDGES)
AL2O3_AXIS_LIMITS = (AL2O3_MIN, AL2O3_MAX)
AL2O3_TRANSECT_HIST_XMAX = max(np.histogram(AMPHIBOLE_EQUAL_WEIGHT_VALUES[pool], bins=AL2O3_BINS)[0].max() for pool in AMPHIBOLE_POOL_ORDER) * 1.05
AL2O3_MINGLED_HIST_XMAX = max(np.histogram(AMPHIBOLE_MINGLED_VALUES[pool][texture], bins=AL2O3_BINS)[0].max() for pool in AMPHIBOLE_POOL_ORDER for texture in ["Light","Dark"]) * 1.05

# Thermobar 1.0.60 needs a float-dtype compatibility patch under pandas 3.
if Version(pd.__version__) >= Version("3.0"):
    for function_name in ["calculate_sites_ridolfi","get_amp_sites_leake","get_amp_sites_avferric_zhang"]:
        original_function = getattr(tb_core, function_name)
        try:
            source_text = inspect.getsource(original_function)
        except OSError:
            continue
        patched_source = re.sub(r"(norm_cations\[['\"][^'\"]+['\"]\]\s*)=0(?![.\d])", r"\1=0.0", source_text)
        if patched_source != source_text:
            exec(patched_source, tb_core.__dict__)
    tb_amphibole.calculate_sites_ridolfi = tb_core.calculate_sites_ridolfi
    tb_amphibole.get_amp_sites_avferric_zhang = tb_core.get_amp_sites_avferric_zhang

line_site_source = amphibole_linescan_points.copy()
for oxide, raw_column in RAW_OXIDE_MAP.items():
    line_site_source[f"_{oxide}_Thermobar"] = pd.to_numeric(line_site_source[raw_column], errors="coerce")
line_required_inputs = [f"_{oxide}_Thermobar" for oxide in RAW_OXIDE_MAP]
line_site_source = line_site_source.loc[line_site_source[line_required_inputs].notna().all(axis=1)].reset_index(drop=True)
line_amp_comps = pd.DataFrame({f"{oxide}_Amp":line_site_source[f"_{oxide}_Thermobar"].astype(float) for oxide in RAW_OXIDE_MAP})
line_amp_comps["Cr2O3_Amp"] = 0.0
line_amp_comps["F_Amp"] = 0.0
line_amp_comps["Cl_Amp"] = pd.to_numeric(line_site_source["Cl Ox%"], errors="coerce").fillna(0.0).astype(float)
line_amp_comps["Sample_ID_Amp"] = [f"{line_id}__row_{index}" for index,line_id in enumerate(line_site_source["Linescan_ID"].astype(str))]
line_ridolfi_sites = tb_core.calculate_sites_ridolfi(line_amp_comps).reset_index(drop=True)
line_site_results = line_site_source[["Linescan_ID","Source_CSV","Sample_Group","_Na2O_Thermobar"]].copy()
line_site_results["Al_VI_C"] = pd.to_numeric(line_ridolfi_sites["Al_VI_C"], errors="coerce")
line_site_results["Al_IV_T"] = pd.to_numeric(line_ridolfi_sites["Al_IV_T"], errors="coerce")
line_site_results = line_site_results.rename(columns={"_Na2O_Thermobar":"Measured_Na2O"}).dropna(subset=["Measured_Na2O","Al_VI_C","Al_IV_T"]).reset_index(drop=True)
line_site_results["Pool"] = np.where(line_site_results["Sample_Group"].isin(AMPHIBOLE_POOL_MEMBERS["B"]), "B", "G")
line_site_results["High_Al2O3"] = line_site_results["Linescan_ID"].isin(HIGH_AL2O3_LINE_IDS)
line_site_results["Plot_Color"] = line_site_results["Linescan_ID"].map(TRANSECT_COLORS)

mingled_site_source = mingled_amphibole_plot_data.copy()
for oxide, raw_column in MINGLED_RAW_OXIDE_MAP.items():
    mingled_site_source[f"_{oxide}_Thermobar"] = pd.to_numeric(mingled_site_source[raw_column], errors="coerce")
mingled_required_inputs = [f"_{oxide}_Thermobar" for oxide in MINGLED_RAW_OXIDE_MAP]
mingled_site_source = mingled_site_source.loc[mingled_site_source[mingled_required_inputs].notna().all(axis=1)].reset_index(drop=True)
mingled_amp_comps = pd.DataFrame({f"{oxide}_Amp":mingled_site_source[f"_{oxide}_Thermobar"].astype(float) for oxide in MINGLED_RAW_OXIDE_MAP})
mingled_amp_comps["F_Amp"] = 0.0
mingled_amp_comps["Cl_Amp"] = 0.0
mingled_amp_comps["Sample_ID_Amp"] = [f"mingled__row_{index}" for index in range(len(mingled_amp_comps))]
mingled_ridolfi_sites = tb_core.calculate_sites_ridolfi(mingled_amp_comps).reset_index(drop=True)
mingled_site_results = mingled_site_source[["Dataset","_display_texture","_Na2O_Thermobar"]].copy()
mingled_site_results["Al_VI_C"] = pd.to_numeric(mingled_ridolfi_sites["Al_VI_C"], errors="coerce")
mingled_site_results["Al_IV_T"] = pd.to_numeric(mingled_ridolfi_sites["Al_IV_T"], errors="coerce")
mingled_site_results = mingled_site_results.rename(columns={"_Na2O_Thermobar":"Measured_Na2O"}).dropna(subset=["Measured_Na2O","Al_VI_C","Al_IV_T"]).reset_index(drop=True)
mingled_site_results["Pool"] = mingled_site_results["Dataset"].map({"74Ming":"B","76Ming":"G"})
assert line_site_results["Plot_Color"].notna().all()
assert set(line_site_results.loc[line_site_results["High_Al2O3"],"Plot_Color"]) == {HIGH_AL2O3_AMPHIBOLE_COLOR}


## 6. Plot and export Figure 6


In [6]:
figure = plt.figure(figsize=(FIGURE_WIDTH_IN, FIGURE_HEIGHT_IN))
outer_grid = figure.add_gridspec(2,1,height_ratios=(2.05,1.45),hspace=0.30)
upper_grid = outer_grid[0].subgridspec(2,3,width_ratios=(4.2,1.40,1.40),wspace=0.10,hspace=0.18)
lower_grid = outer_grid[1].subgridspec(1,2,wspace=0.22)
upper_axes = np.empty((2,3),dtype=object)
panel_labels = []
panel_number = 0

for row,pool in enumerate(AMPHIBOLE_POOL_ORDER):
    ax_line = figure.add_subplot(upper_grid[row,0])
    ax_hist = figure.add_subplot(upper_grid[row,1],sharey=ax_line)
    ax_mingled = figure.add_subplot(upper_grid[row,2],sharey=ax_line)
    upper_axes[row] = (ax_line,ax_hist,ax_mingled)
    transects = AMPHIBOLE_TRANSECTS[pool]
    for transect in transects:
        linescan_id = transect.attrs["linescan_id"]
        ax_line.plot(transect["_distance_um"].to_numpy(float),transect["_Al2O3_norm"].to_numpy(float),
                     "-^",color=TRANSECT_COLORS[linescan_id],linewidth=0.85,markersize=1.80,
                     markeredgewidth=0.0,alpha=0.90,zorder=4 if linescan_id in HIGH_AL2O3_LINE_IDS else 2)
    transect_values = AMPHIBOLE_EQUAL_WEIGHT_VALUES[pool]
    transect_median = float(np.median(transect_values))
    transect_color = AMPHIBOLE_POOL_HIST_COLORS[pool]
    ax_hist.hist(transect_values,bins=AL2O3_BINS,orientation="horizontal",color=transect_color,edgecolor=transect_color,alpha=0.60,linewidth=0.85)
    ax_hist.axhline(transect_median,color=transect_color,linestyle="--",linewidth=0.85)
    ax_hist.text(0.97,0.90,pool,transform=ax_hist.transAxes,ha="right",va="top",fontsize=6.0,color=transect_color)
    ax_hist.text(0.97,0.76,f"n={len(transects)}\nmed={transect_median:.1f}",transform=ax_hist.transAxes,ha="right",va="top",fontsize=6.0,color=transect_color,bbox={"facecolor":"white","edgecolor":"none","alpha":0.76,"pad":0.35})
    light_values = AMPHIBOLE_MINGLED_VALUES[pool]["Light"]
    dark_values = AMPHIBOLE_MINGLED_VALUES[pool]["Dark"]
    light_color = MINGLED_AMPHIBOLE_COLORS[pool]["Light"]
    dark_color = MINGLED_AMPHIBOLE_COLORS[pool]["Dark"]
    ax_mingled.hist(light_values,bins=AL2O3_BINS,orientation="horizontal",color=light_color,edgecolor=light_color,alpha=0.60,linewidth=0.85)
    ax_mingled.hist(dark_values,bins=AL2O3_BINS,orientation="horizontal",histtype="step",color=dark_color,linewidth=0.85)
    ax_mingled.axhline(float(np.median(light_values)),color=light_color,linestyle="--",linewidth=0.85)
    ax_mingled.axhline(float(np.median(dark_values)),color=dark_color,linestyle=":",linewidth=0.95)
    ax_mingled.legend(handles=[
        Patch(facecolor=light_color,edgecolor=light_color,alpha=0.60,label=f"Light n={len(light_values)}\nmed={np.median(light_values):.1f}"),
        Patch(facecolor="none",edgecolor=dark_color,linewidth=0.85,label=f"Dark n={len(dark_values)}\nmed={np.median(dark_values):.1f}"),
    ],loc="lower right" if pool=="B" else "upper right",fontsize=6.0,frameon=True,framealpha=1.0,borderpad=0.0,labelspacing=0.16,handlelength=0.95,handletextpad=0.28)
    for axis in [ax_line,ax_hist,ax_mingled]:
        axis.set_ylim(*AL2O3_AXIS_LIMITS)
        axis.grid(True,axis="y",color="0.92",linewidth=0.35,zorder=0)
        axis.set_axisbelow(True)
        for spine in axis.spines.values():
            spine.set_visible(True); spine.set_color("black"); spine.set_linewidth(0.50)
        panel_number += 1
        panel_label = axis.text(0.015,0.985,f"{chr(64+panel_number)})",transform=axis.transAxes,ha="left",va="top",fontsize=PANEL_LABEL_SIZE_PT,fontweight="bold",bbox={"facecolor":"white","edgecolor":"none","alpha":0.82,"pad":0.35},zorder=30)
        panel_labels.append(panel_label)
    ax_line.set_xlim(left=0)
    ax_line.text(0.985,0.88,f"Sample {pool}\n{len(transects)} transects; {sum(len(frame) for frame in transects):,} points",transform=ax_line.transAxes,ha="right",va="top",fontsize=6.0,bbox={"facecolor":"white","edgecolor":"none","alpha":0.78,"pad":0.6},zorder=20)
    ax_line.ticklabel_format(axis="x",style="plain",useOffset=False)
    ax_hist.set_xlim(0,AL2O3_TRANSECT_HIST_XMAX)
    ax_mingled.set_xlim(0,AL2O3_MINGLED_HIST_XMAX)
    for histogram_axis in [ax_hist,ax_mingled]:
        histogram_axis.tick_params(axis="y",left=True,right=False,labelleft=False,labelright=False,length=2.5,width=0.5)
        histogram_axis.tick_params(axis="x",bottom=False,top=False,labelbottom=False,labeltop=False)
    if row == 0:
        ax_line.set_title("Main-pumice amphibole transects",pad=4)
        ax_hist.set_title("Main pumice",pad=4)
        ax_mingled.set_title("Mingled pumice point analyses",pad=4)
        ax_line.tick_params(axis="x",labelbottom=False)
    else:
        ax_line.set_xlabel("Distance (µm)")
        x_min,x_max = ax_line.get_xlim()
        for label,x_position in [("Rim",x_min),("Core",x_max)]:
            ax_line.annotate(label,xy=(x_position,0),xycoords=("data","axes fraction"),xytext=(0,-13),textcoords="offset points",ha="center",va="top",fontsize=6.0,fontstyle="italic")

ax_g = figure.add_subplot(lower_grid[0,0])
ax_h = figure.add_subplot(lower_grid[0,1])
site_axes = (ax_g,ax_h)
for axis,site_variable,ylabel in zip(site_axes,("Al_VI_C","Al_IV_T"),(r"Al on C site (apfu)",r"Al on T site (apfu)")):
    for linescan_id,points in line_site_results.groupby("Linescan_ID",sort=True):
        is_high = linescan_id in HIGH_AL2O3_LINE_IDS
        axis.scatter(points["Measured_Na2O"],points[site_variable],s=TRANSECT_SITE_POINT_SIZE,marker=TRANSECT_SITE_MARKER,color=TRANSECT_COLORS[linescan_id],edgecolors="black",linewidths=AMPHIBOLE_POINT_EDGE_WIDTH,alpha=0.92 if is_high else 0.62,zorder=5 if is_high else 2)
    for pool in AMPHIBOLE_POOL_ORDER:
        for texture in ["Light","Dark"]:
            points = mingled_site_results.loc[mingled_site_results["Pool"].eq(pool) & mingled_site_results["_display_texture"].eq(texture)]
            axis.scatter(points["Measured_Na2O"],points[site_variable],s=MINGLED_SITE_POINT_SIZE,marker=MINGLED_SITE_MARKER,color=MINGLED_AMPHIBOLE_COLORS[pool][texture],edgecolors="black",linewidths=AMPHIBOLE_POINT_EDGE_WIDTH,alpha=0.72,zorder=6)
    axis.set_xlabel(r"Na$_2$O (wt.%)")
    axis.set_ylabel(ylabel)
    axis.grid(True,color="0.92",linewidth=0.35,zorder=0)
    axis.set_axisbelow(True)
    for spine in axis.spines.values():
        spine.set_visible(True); spine.set_color("black"); spine.set_linewidth(0.50)
    panel_number += 1
    panel_label = axis.text(0.015,0.985,f"{chr(64+panel_number)})",transform=axis.transAxes,ha="left",va="top",fontsize=PANEL_LABEL_SIZE_PT,fontweight="bold",bbox={"facecolor":"white","edgecolor":"none","alpha":0.82,"pad":0.35},zorder=30)
    panel_labels.append(panel_label)

figure.text(0.022,0.715,r"Al$_2$O$_3$ (wt.%)",rotation="vertical",ha="center",va="center",fontsize=AXIS_LABEL_SIZE_PT)
legend_handles = [
    Patch(facecolor=AMPHIBOLE_POOL_HIST_COLORS["B"],edgecolor="none",label="B main-pumice transects"),
    Patch(facecolor=AMPHIBOLE_POOL_HIST_COLORS["G"],edgecolor="none",label="G main-pumice transects"),
    Patch(facecolor=MINGLED_AMPHIBOLE_COLORS["B"]["Light"],edgecolor="none",label="B light mingled"),
    Patch(facecolor=MINGLED_AMPHIBOLE_COLORS["B"]["Dark"],edgecolor="none",label="B dark mingled"),
    Patch(facecolor=MINGLED_AMPHIBOLE_COLORS["G"]["Light"],edgecolor="none",label="G light mingled"),
    Patch(facecolor=MINGLED_AMPHIBOLE_COLORS["G"]["Dark"],edgecolor="none",label="G dark mingled"),
    Patch(facecolor=HIGH_AL2O3_AMPHIBOLE_COLOR,edgecolor="none",label="High-Al2O3 trio"),
    Line2D([0],[0],marker="^",linestyle="none",markersize=5.5,markerfacecolor="0.45",markeredgecolor="black",markeredgewidth=AMPHIBOLE_POINT_EDGE_WIDTH,label="Amphibole analyses"),
]
figure.legend(handles=legend_handles,title="Samples, context, and data type",loc="lower center",bbox_to_anchor=(0.53,0.015),ncol=3,frameon=True,framealpha=1.0,edgecolor="black",fancybox=False,fontsize=6.0,title_fontsize=6.0,borderaxespad=0,borderpad=0.35,columnspacing=1.0,labelspacing=0.22,handleheight=0.8)
figure.subplots_adjust(left=0.078,right=0.990,bottom=0.17,top=0.945)

FIGURE_STEM = "Figure_06_Amphiboles"
EXPORT_PATHS = {}
for output_format in ["pdf","png","svg"]:
    output_path = OUTPUT_DIRECTORY / f"{FIGURE_STEM}.{output_format}"
    save_options = {"format":output_format,"facecolor":"white","transparent":False}
    if output_format == "png":
        save_options["dpi"] = EXPORT_DPI
    figure.savefig(output_path,**save_options)
    if not output_path.is_file() or output_path.stat().st_size == 0:
        raise IOError(f"Expected export was not created: {output_path}")
    EXPORT_PATHS[output_format] = output_path
plt.close(figure)


## 7. Validate the populations, marker encoding, and exports


In [7]:
assert panel_number == 8
assert len(AMPHIBOLE_TRANSECTS["B"]) > 0 and len(AMPHIBOLE_TRANSECTS["G"]) > 0
assert len(mingled_amphibole_plot_data) > 0
assert TRANSECT_SITE_MARKER == "^"
assert MINGLED_SITE_MARKER == "^"
assert set(line_site_results.loc[line_site_results["High_Al2O3"],"Plot_Color"]) == {HIGH_AL2O3_AMPHIBOLE_COLOR}
assert tuple(round(value,8) for value in figure.get_size_inches()) == (7.25,5.15)
assert all(spine.get_visible() for axis in figure.axes for spine in axis.spines.values())
assert len(panel_labels) == 8
assert [label.get_text() for label in panel_labels] == ["A)","B)","C)","D)","E)","F)","G)","H)"]
assert all(path.is_file() and path.stat().st_size > 0 for path in EXPORT_PATHS.values())
with Image.open(EXPORT_PATHS["png"]) as image:
    assert image.size == (2175,1545)
svg_text = EXPORT_PATHS["svg"].read_text(encoding="utf-8",errors="replace")
assert "<text" in svg_text and "<image" not in svg_text
assert b"/Type3" not in EXPORT_PATHS["pdf"].read_bytes()
SOURCE_HASHES_AFTER = {
    "S1":hashlib.sha256(S1_PATH.read_bytes()).hexdigest(),
    "S2":hashlib.sha256(S2_PATH.read_bytes()).hexdigest(),
    "S3":hashlib.sha256(S3_PATH.read_bytes()).hexdigest(),
}
assert SOURCE_HASHES_AFTER == SOURCE_HASHES_BEFORE
print("Figure 6 complete.")
print("Main-pumice transects and mingled-pumice point analyses are explicitly separated.")
print("Outputs:", {key:str(value) for key,value in EXPORT_PATHS.items()})


Figure 6 complete.
Main-pumice transects and mingled-pumice point analyses are explicitly separated.
Outputs: {'pdf': '/Users/annikadechert/Library/Mobile Documents/com~apple~CloudDocs/!!!Cal_Postdoc/Berkeley Postdoc/Glacier Peak Efforts/SA_MI_Draft_FinalFigures_Work/Nov6_AllFigures/Figure_06_Amphiboles/Figure_06_Amphiboles_Final/Figure_06_Amphiboles.pdf', 'png': '/Users/annikadechert/Library/Mobile Documents/com~apple~CloudDocs/!!!Cal_Postdoc/Berkeley Postdoc/Glacier Peak Efforts/SA_MI_Draft_FinalFigures_Work/Nov6_AllFigures/Figure_06_Amphiboles/Figure_06_Amphiboles_Final/Figure_06_Amphiboles.png', 'svg': '/Users/annikadechert/Library/Mobile Documents/com~apple~CloudDocs/!!!Cal_Postdoc/Berkeley Postdoc/Glacier Peak Efforts/SA_MI_Draft_FinalFigures_Work/Nov6_AllFigures/Figure_06_Amphiboles/Figure_06_Amphiboles_Final/Figure_06_Amphiboles.svg'}
